## Super Resolution Dataset Splitting

The dataset is divided into training, validation, and testing sets using
a stratified split with a ratio of 70:20:10.

Stratified splitting is applied to preserve the class distribution across
the three subsets.

In [ ]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

# =========================
# CONFIGURATION
# =========================
SOURCE_DIR = Path("path/to/dataset")
OUTPUT_DIR = Path("path/to/output")

TRAIN_DIR = OUTPUT_DIR / "train"
VAL_DIR = OUTPUT_DIR / "val"
TEST_DIR = OUTPUT_DIR / "test"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}
RANDOM_STATE = 42

# =========================
# COLLECT IMAGE PATHS
# =========================
image_paths = []
labels = []

for class_dir in sorted(SOURCE_DIR.iterdir()):
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name

    for image_path in sorted(class_dir.iterdir()):
        if image_path.suffix.lower() in VALID_EXTENSIONS:
            image_paths.append(Path(class_name) / image_path.name)
            labels.append(class_name)

print(f"Total images: {len(image_paths)}")

# =========================
# SPLIT DATA: 70% / 20% / 10%
# =========================
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=1/3,
    random_state=RANDOM_STATE,
    stratify=temp_labels
)

# =========================
# CREATE OUTPUT DIRECTORIES
# =========================
for output_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    for class_dir in sorted(SOURCE_DIR.iterdir()):
        if class_dir.is_dir():
            (output_dir / class_dir.name).mkdir(
                parents=True,
                exist_ok=True
            )

# =========================
# COPY IMAGES
# =========================
def copy_images(file_paths, target_root):
    for relative_path in file_paths:
        source_path = SOURCE_DIR / relative_path
        target_path = target_root / relative_path

        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)

copy_images(train_paths, TRAIN_DIR)
copy_images(val_paths, VAL_DIR)
copy_images(test_paths, TEST_DIR)

# =========================
# DISPLAY SPLIT SUMMARY
# =========================
total_images = len(image_paths)

print("\n=== DATASET SPLIT ===")
print(f"Train      : {len(train_paths):,} ({len(train_paths) / total_images * 100:.1f}%)")
print(f"Validation : {len(val_paths):,} ({len(val_paths) / total_images * 100:.1f}%)")
print(f"Test       : {len(test_paths):,} ({len(test_paths) / total_images * 100:.1f}%)")

## Classification Dataset Splitting

The classification dataset is divided into training, validation, and testing subsets with a ratio of 70:15:15 using stratified sampling to preserve the class distribution across all subsets.

In [ ]:
import shutil
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

# =====================================================
# CONFIGURATION
# =====================================================
SEED = 42

SOURCE_DIR = Path("path/to/dataset")
OUTPUT_DIR = Path("path/to/split_dataset")

CLASSES = ["healthy", "frass", "egg", "larva"]
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# Remove previous output to prevent mixed results when rerunning
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =====================================================
# COLLECT IMAGE PATHS
# =====================================================
all_images = []

for class_name in CLASSES:
    class_dir = SOURCE_DIR / class_name

    if not class_dir.exists():
        raise FileNotFoundError(
            f"Class directory not found: {class_dir}"
        )

    for image_path in sorted(class_dir.iterdir()):
        if image_path.suffix.lower() in VALID_EXTENSIONS:
            all_images.append(
                (class_name, image_path)
            )

print(f"Total images: {len(all_images):,}")

initial_distribution = Counter(
    label for label, _ in all_images
)

print("\nInitial class distribution:")
for class_name in CLASSES:
    print(
        f"{class_name:<10}: "
        f"{initial_distribution[class_name]:,}"
    )


# =====================================================
# STRATIFIED SPLIT: 70% TRAIN / 15% VAL / 15% TEST
# =====================================================
train_val, test = train_test_split(
    all_images,
    test_size=0.15,
    random_state=SEED,
    stratify=[label for label, _ in all_images]
)

train, val = train_test_split(
    train_val,
    test_size=0.15 / 0.85,
    random_state=SEED,
    stratify=[label for label, _ in train_val]
)


# =====================================================
# COPY IMAGES TO SPLIT DIRECTORIES
# =====================================================
splits = {
    "train": train,
    "val": val,
    "test": test
}

for split_name, split_data in splits.items():

    for label, source_path in split_data:

        destination_dir = (
            OUTPUT_DIR / split_name / label
        )

        destination_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        destination_path = (
            destination_dir /
            f"{label}_{source_path.name}"
        )

        shutil.copy2(
            source_path,
            destination_path
        )


# =====================================================
# SPLIT SUMMARY
# =====================================================
print("\n=== DATASET SPLIT SUMMARY ===")

total_images = len(all_images)

for split_name, split_data in splits.items():

    distribution = Counter(
        label for label, _ in split_data
    )

    percentage = (
        len(split_data) /
        total_images *
        100
    )

    print(
        f"\n{split_name.upper()}: "
        f"{len(split_data):,} images "
        f"({percentage:.1f}%)"
    )

    for class_name in CLASSES:
        print(
            f"  {class_name:<10}: "
            f"{distribution[class_name]:,}"
        )

print(
    f"\nTotal images: "
    f"{sum(len(data) for data in splits.values()):,}"
)

print(
    f"Output directory: {OUTPUT_DIR}"
)

: 

## Patch Extraction

The training and validation images are divided into non-overlapping patches of size 80 × 80 pixels. A stride equal to the patch size is used to ensure that each patch does not overlap with the others.

In [ ]:
import cv2
from pathlib import Path

# =========================
# CONFIGURATION
# =========================
PATCH_SIZE = 80
STRIDE = PATCH_SIZE  # No overlap

TRAIN_DIR = Path("path/to/train")
VAL_DIR = Path("path/to/val")

OUTPUT_DIR = Path("path/to/patches")
TRAIN_OUTPUT_DIR = OUTPUT_DIR / "train"
VAL_OUTPUT_DIR = OUTPUT_DIR / "val"

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# =========================
# PATCH EXTRACTION
# =========================
def extract_patches(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    total_patches = 0

    for class_dir in sorted(input_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        class_output_dir = output_dir / class_dir.name
        class_output_dir.mkdir(parents=True, exist_ok=True)

        class_patch_count = 0

        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() not in VALID_EXTENSIONS:
                continue

            image = cv2.imread(str(image_path))

            if image is None:
                continue

            height, width = image.shape[:2]

            patch_index = 0

            for y in range(0, height - PATCH_SIZE + 1, STRIDE):
                for x in range(0, width - PATCH_SIZE + 1, STRIDE):

                    patch = image[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE
                    ]

                    patch_name = (
                        f"{image_path.stem}_{patch_index}.jpg"
                    )

                    cv2.imwrite(
                        str(class_output_dir / patch_name),
                        patch
                    )

                    patch_index += 1
                    class_patch_count += 1
                    total_patches += 1

        print(
            f"{class_dir.name}: "
            f"{class_patch_count:,} patches"
        )

    return total_patches


# =========================
# EXECUTION
# =========================
print("Extracting training patches...")
train_patch_count = extract_patches(
    TRAIN_DIR,
    TRAIN_OUTPUT_DIR
)

print("\nExtracting validation patches...")
val_patch_count = extract_patches(
    VAL_DIR,
    VAL_OUTPUT_DIR
)

print("\n=== PATCH EXTRACTION SUMMARY ===")
print(f"Training patches   : {train_patch_count:,}")
print(f"Validation patches : {val_patch_count:,}")
print(f"Total patches      : {train_patch_count + val_patch_count:,}")

## Generation of LR-HR Image Pairs

High-resolution (HR) images are converted into low-resolution (LR) images using bicubic downsampling. The LR images are then upsampled to the original HR dimensions using nearest-neighbor interpolation.

Two super-resolution scales are evaluated:

- Scale ×2
- Scale ×4

The resulting LR and LR-upsampled images are used as inputs for the RCAN super-resolution experiments.

In [ ]:
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

# =========================
# CONFIGURATION
# =========================
PATCH_DIR = Path("path/to/patches")
TEST_DIR = Path("path/to/test")

OUTPUT_DIR = Path("path/to/sr_data")

SPLIT_DIRS = {
    "train": PATCH_DIR / "train",
    "val": PATCH_DIR / "val",
    "test": TEST_DIR
}

SCALES = [2, 4]
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# =========================
# PROCESS IMAGE SPLIT
# =========================
def process_split(input_dir, output_dir, split_name, scale):
    print(f"\nProcessing {split_name} - Scale ×{scale}")

    total_images = 0

    for class_dir in sorted(input_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        lr_dir = output_dir / split_name / f"x{scale}" / "LR" / class_dir.name
        lr_up_dir = output_dir / split_name / f"x{scale}" / "LR_up" / class_dir.name

        lr_dir.mkdir(parents=True, exist_ok=True)
        lr_up_dir.mkdir(parents=True, exist_ok=True)

        class_count = 0

        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() not in VALID_EXTENSIONS:
                continue

            hr_image = Image.open(image_path).convert("RGB")
            width, height = hr_image.size

            # Downsampling: HR → LR
            downsample = transforms.Resize(
                (height // scale, width // scale),
                interpolation=InterpolationMode.BICUBIC
            )
            lr_image = downsample(hr_image)

            # Upsampling: LR → LR_up
            upsample = transforms.Resize(
                (height, width),
                interpolation=InterpolationMode.NEAREST
            )
            lr_up_image = upsample(lr_image)

            # Save LR and LR_up images
            lr_image.save(lr_dir / image_path.name)
            lr_up_image.save(lr_up_dir / image_path.name)

            class_count += 1
            total_images += 1

        print(
            f"{class_dir.name}: "
            f"{class_count:,} images"
        )

    print(
        f"{split_name} - Scale ×{scale}: "
        f"{total_images:,} images processed"
    )


# =========================
# GENERATE LR-HR PAIRS
# =========================
for scale in SCALES:
    print(f"\n{'=' * 50}")
    print(f"GENERATING DATA FOR SCALE ×{scale}")
    print(f"{'=' * 50}")

    for split_name, split_dir in SPLIT_DIRS.items():
        process_split(
            split_dir,
            OUTPUT_DIR,
            split_name,
            scale
        )

print("\nLR-HR pair generation completed.")